# 03. ????????? Protocol-Fixed Efficient HADES ??????

**???? ???????.** ??????? ???????????? ????????? HADES-like fusion-??????, ???????? ?? ? baseline ?? `02`, ?????????????? dashboard bundle ? ????????????? ???????????. ????????????????? ?????? ?????? ?? ???????? ???????? ?????????.

????????????? ???? ???????????: `logs -> cached embeddings -> log encoder`, `KPI time-series segments -> metric encoder`, ????? bidirectional cross-modal attention ? multi-head detection ?? ??????????.


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
ARTIFACT_DIR = PROJECT_ROOT / "dashboard" / "artifacts"
CONTRACT_PATH = ARTIFACT_DIR / "contract" / "strict_intensity_contract.csv.gz"
REPORT_DIR = ARTIFACT_DIR / "reports"
BASELINE_DIR = REPORT_DIR / "protocol_fixed_baselines"
MODEL_OUT_DIR = ARTIFACT_DIR / "model" / "protocol_fixed_efficient_hades"
RUN_ROOT = ARTIFACT_DIR / "model_runs" / "protocol_fixed_hades"
RUN_ID = "final_protocol_fixed_hades"
BUNDLE_DIR = ARTIFACT_DIR / "bundle"

def run_command(command: list[str]) -> None:
    print(" ".join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

def read_json(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))

def display_existing(path: Path, reader=pd.read_csv, **kwargs):
    if not path.exists():
        print(f"???? ???? ?? ??????: {path.relative_to(PROJECT_ROOT)}")
        return pd.DataFrame()
    frame = reader(path, **kwargs)
    display(frame)
    return frame


## 1. ???????????? ?????????? ????????

**???? ???????.** ????????????? ???????????? production-????????????. ??? ??????? ???????? protocol-fixed ??????????: `stratified_by_workload`, `workload_norm`, `raw_plus_deviation`, multi-head targets.


In [ ]:
FINAL_CONFIG = {
    "run_id": RUN_ID,
    "device": "cuda",
    "split_protocol": "stratified_by_workload",
    "variant": "modern_metric",
    "seed": 43,
    "monitor_target": "horizon_60s",
    "alert_target": "horizon_60s",
    "normalization": "workload_norm",
    "metric_feature_mode": "raw_plus_deviation",
    "epochs": 30,
    "batch_size": 128,
    "fp_budget": 0.10,
    "quantization": "dynamic",
}
FINAL_CONFIG


**????????? ?????.** `horizon_60s` ???????? ??????????? ??????? ??? ??????? alert-layer, ? `horizon_120s/300s` ???????????? ? dashboard ??? ????? ?????????? early-warning scores.


## 2. ???????? ? export

**???? ???????.** ????????? ????????? ???????? ? ????? ???????????? artifacts ??? dashboard. ONNX/INT8 ?????????????? ??? runtime path ??? ??????? CPU, ?? ???????? ??????????? ?? ???????? Torch/FP32 ??????.


In [ ]:
TRAIN_FINAL_MODEL = True
train_command = [
    sys.executable, "-m", "dashboard.training.final_model",
    "--run-id", FINAL_CONFIG["run_id"],
    "--device", FINAL_CONFIG["device"],
    "--run-root", str(RUN_ROOT),
    "--contract-path", str(CONTRACT_PATH),
    "--model-out-dir", str(MODEL_OUT_DIR),
    "--bundle-out-dir", str(BUNDLE_DIR),
    "--split-protocol", FINAL_CONFIG["split_protocol"],
    "--variant", FINAL_CONFIG["variant"],
    "--seed", str(FINAL_CONFIG["seed"]),
    "--monitor-target", FINAL_CONFIG["monitor_target"],
    "--alert-target", FINAL_CONFIG["alert_target"],
    "--normalization", FINAL_CONFIG["normalization"],
    "--metric-feature-mode", FINAL_CONFIG["metric_feature_mode"],
    "--epochs", str(FINAL_CONFIG["epochs"]),
    "--batch-size", str(FINAL_CONFIG["batch_size"]),
    "--fp-budget", str(FINAL_CONFIG["fp_budget"]),
    "--quantization", FINAL_CONFIG["quantization"],
]
if TRAIN_FINAL_MODEL:
    run_command(train_command)
else:
    print("Final model training skipped; existing artifacts will be read.")


In [ ]:
# ????? ????????? export ??????? ????? ???????? ??? ????????? artifacts ?? ?????? ??????.
export_command = [
    sys.executable, "-m", "dashboard.export.bundle",
    "--mode", "protocol_fixed",
    "--protocol-run-dir", str(RUN_ROOT / RUN_ID),
    "--contract-path", str(CONTRACT_PATH),
    "--out-dir", str(BUNDLE_DIR),
    "--protocol-model-out-dir", str(MODEL_OUT_DIR),
    "--split-protocol", FINAL_CONFIG["split_protocol"],
    "--variant", FINAL_CONFIG["variant"],
    "--seed", str(FINAL_CONFIG["seed"]),
]
run_command(export_command)


**????????? ?????.** ????? ????? ???? dashboard ?????? ???????? ?????? ?? `dashboard/artifacts/bundle/*`. ????????? hypothesis outputs ?????? ?? ???????? ?????? ?????????? ????????.


## 3. ??????? ????????? ??????

**???? ???????.** ?????????? window-level ???????? ?? ???? target ? ???????? ? baseline ?? `02`.


In [ ]:
run_dir = RUN_ROOT / RUN_ID
final_metrics_path = run_dir / "comparison_metrics.csv"
final_metrics = pd.read_csv(final_metrics_path)
final_test = final_metrics.query("split == 'test' and runtime == 'torch_fp32' and threshold_policy == 'validation_best_f1'").copy()
final_test = final_test[(final_test["split_protocol"] == FINAL_CONFIG["split_protocol"]) & (final_test["variant"] == FINAL_CONFIG["variant"]) & (final_test["seed"] == FINAL_CONFIG["seed"])]
for col in ["average_precision", "roc_auc", "precision", "recall", "f1", "false_positives", "missed_anomalies"]:
    if col in final_test.columns:
        final_test[col] = pd.to_numeric(final_test[col], errors="coerce")
display(final_test[["target", "precision", "recall", "f1", "average_precision", "roc_auc", "false_positives", "missed_anomalies"]].round(4))


In [ ]:
baseline_metrics_path = BASELINE_DIR / "baseline_protocol_metrics.csv"
baseline_metrics = pd.read_csv(baseline_metrics_path) if baseline_metrics_path.exists() else pd.DataFrame()
if not baseline_metrics.empty:
    baseline_test = baseline_metrics.query("split == 'test' and threshold_policy == 'validation_best_f1'").copy()
    best_baseline = baseline_test.sort_values(["target", "f1"], ascending=[True, False]).groupby("target", as_index=False).head(1)
    final_compare = final_test.assign(model="Efficient HADES")[["target", "model", "precision", "recall", "f1", "average_precision", "roc_auc"]]
    base_compare = best_baseline.assign(model="?????? baseline")[["target", "model", "precision", "recall", "f1", "average_precision", "roc_auc"]]
    comparison = pd.concat([base_compare, final_compare], ignore_index=True)
    display(comparison.sort_values(["target", "model"]).round(4))
    plot = comparison[comparison["target"].isin(["horizon_60s", "horizon_120s", "horizon_300s", "y_warning", "y_risk"])]
    ax = plot.pivot(index="target", columns="model", values="f1").plot(kind="bar", figsize=(10, 4), title="F1: ????????? ?????? ?????? baseline")
    ax.set_xlabel("Target")
    ax.set_ylabel("F1")
    ax.set_ylim(0, 1.0)
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
else:
    print("Baseline artifacts ???????????; ??????? ????????? 02_baseline_pipeline.ipynb.")


**????????? ?????.** ????????? ??????? ?????? ??? ?????????? protocol. ???? baseline ?????????? ?? ????????? horizon, ??? ?? ??????: ?? ?????????? ??????? ??????????? ??????, ? ????????? ?????? ????? ??? ??????????????? ?????? ? alert-layer ? ??????????? dashboard bundle.


## 4. Alert-level ??????

**???? ???????.** ??????? ?? ????????? ???? ? ????????????????? ??????: ??????? ?????????? ??????????, ??????? ????????? ? ??????? ?????? alert-??????? ???????.


In [ ]:
alert_metrics_path = run_dir / "alert_level_metrics.csv"
alert_metrics = pd.read_csv(alert_metrics_path) if alert_metrics_path.exists() else pd.DataFrame()
if not alert_metrics.empty:
    selected = alert_metrics[
        (alert_metrics["split_protocol"] == FINAL_CONFIG["split_protocol"]) &
        (alert_metrics["variant"] == FINAL_CONFIG["variant"]) &
        (alert_metrics["seed"] == FINAL_CONFIG["seed"])
    ].copy()
    display(selected.head(40))
else:
    print("Alert-level metrics ???? ?? ???????.")


**????????? ?????.** Alert-layer ????? ??? ???????????? ????????????. ?? ?????????? window-level ??? ????? consecutive windows ? cooldown, ?? ?????? ?????? ???? ??????? ?? `model_score_raw` ? ????? incident.


## 5. Dashboard bundle ? runtime slots

**???? ???????.** ?????????, ??? ????????? bundle ?????????? ? ???????? ????????????????? ???? ??? Torch/ONNX/INT8 runtime.


In [ ]:
bundle_files = sorted([p.name for p in BUNDLE_DIR.glob("*") if p.is_file()])
print("Bundle files:")
for name in bundle_files:
    print("-", name)
report = read_json(BUNDLE_DIR / "dashboard_model_report.json")
print(json.dumps({
    "model_name": report.get("model_name"),
    "primary_dashboard_target": report.get("primary_dashboard_target"),
    "secondary_dashboard_target": report.get("secondary_dashboard_target"),
    "runtime": report.get("runtime"),
    "onnx_int8_plan": report.get("onnx_int8_plan"),
}, ensure_ascii=False, indent=2))


**????????? ?????.** ONNX/INT8 ?? ????????? ???????? ??????: ??? runtime-??????????? ??? CPU-only ?????. Dashboard ?????? ???????? ?? Torch-derived bundle ???? ???? INT8 ???? ??? ?? ???????????.


## ???? 03

????????? ????????? ????? ? ???????????? protocol-fixed Efficient HADES ?????? ? ?????????? dashboard bundle. ?????? ??????? ??????????????? ????? `streamlit run dashboard/app.py`, ? notebooks ???????? ???????? ????????????? ????????: ??????, baseline, ??????, ?????????, ???????????.
